In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv


In [2]:
import pandas as pd
import numpy as np
import os

# 1. Dataset ka path dhoondhna
file_path = ""
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        file_path = os.path.join(dirname, filename)
        print("📁 Dataset path mila:", file_path)

# 2. Day 7 Concept: Try/Except se error handling
try:
    print("\nData load ho raha hai... (isme thoda time lag sakta hai)")
    df = pd.read_csv(file_path)
    
    print("✅ Dataset successfully load ho gaya!")
    print("Total Rows & Columns:", df.shape)
    
    print("\n--- Fraud (1) vs Normal (0) Transactions ---")
    print(df['Class'].value_counts())
    
except FileNotFoundError:
    print("❌ Error: File nahi mili! Path check karo.")
except Exception as e:
    print(f"❌ Koi anjaan error aaya: {e}")

📁 Dataset path mila: /kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv

Data load ho raha hai... (isme thoda time lag sakta hai)
✅ Dataset successfully load ho gaya!
Total Rows & Columns: (284807, 31)

--- Fraud (1) vs Normal (0) Transactions ---
Class
0    284315
1       492
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# Day 7 Concept: Function definition with parameters and return values
def prepare_fraud_data(data):
    print("⚙️ Data processing function start ho raha hai...")
    
    # 1. 'Time' aur 'Amount' columns ko scale karna zaroori hai (baaki sab pehle se scaled hain)
    scaler = RobustScaler()
    data['scaled_amount'] = scaler.fit_transform(data['Amount'].values.reshape(-1,1))
    data['scaled_time'] = scaler.fit_transform(data['Time'].values.reshape(-1,1))
    
    # Purane columns hata do aur naye scaled columns aage laga do
    data.drop(['Time','Amount'], axis=1, inplace=True)
    
    # 2. X (Features) aur y (Target) ko alag karna
    X = data.drop('Class', axis=1)
    y = data['Class']
    
    # 3. Train aur Test mein split karna (80% training, 20% testing)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("✅ Data split successful!")
    print(f"Training Features Shape: {X_train.shape}")
    print(f"Testing Features Shape: {X_test.shape}")
    
    # Function se 4 variables return kar rahe hain
    return X_train, X_test, y_train, y_test

# Function ko call karna aur return hui values ko variables mein save karna
X_train, X_test, y_train, y_test = prepare_fraud_data(df)

⚙️ Data processing function start ho raha hai...
✅ Data split successful!
Training Features Shape: (227845, 30)
Testing Features Shape: (56962, 30)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Day 7 Concept: Complex function with error handling for ML Training
def train_fraud_detector(X_train, X_test, y_train, y_test):
    print("🤖 Model training shuru ho rahi hai... (Wait karo)")
    try:
        # class_weight='balanced' is the magic trick for rare frauds!
        model = LogisticRegression(class_weight='balanced', max_iter=1000)
        model.fit(X_train, y_train)
        
        print("✅ Training complete! Ab test data par hackers ko pakadte hain...\n")
        y_pred = model.predict(X_test)
        
        print("--- 📊 Classification Report ---")
        print(classification_report(y_test, y_pred))
        
        print("--- 🕵️ Confusion Matrix ---")
        print("Top-Right: Normal but flagged | Bottom-Left: Fraud but missed")
        print(confusion_matrix(y_test, y_pred))
        
        return model
    except Exception as e:
        print(f"❌ Model training mein error aaya: {e}")
        return None

# Function ko call karna
trained_model = train_fraud_detector(X_train, X_test, y_train, y_test)

🤖 Model training shuru ho rahi hai... (Wait karo)
✅ Training complete! Ab test data par hackers ko pakadte hain...

--- 📊 Classification Report ---
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962

--- 🕵️ Confusion Matrix ---
Top-Right: Normal but flagged | Bottom-Left: Fraud but missed
[[55475  1389]
 [    8    90]]
